In [127]:
# Sign in and set up environment
import requests
import json
def json_result_dict(httpResult):
    return json.loads(httpResult.content.decode())
session = requests.Session()

# Settings
ehrUrl = "http://localhost:9091/ehr"
username = "test"
password = "test"
userObject = {
    'username': username,
    'password': password,
}
tenant = {
    'nameDisplay': 'test'
}
registry = {
    "name":"test script",
    "iisHl7Url":"http://localhost:8080/iis/soap",
    "iisFhirUrl":"http://localhost:8080/iis/fhir",
    "iisFhirMessagingUrl":"http://localhost:8080/iis/fhirMessaging/soap",
    "iisUsername":"test",
    "iisFacilityId":"test",
    "iisPassword":"test",
}

# login
authResult = session.post(f"{ehrUrl}/auth", json=userObject, headers={"Content-Type": "application/json"})
token = json_result_dict(authResult)['accessToken']
authorizationHeader = f"Bearer {token}"
headers = {"Content-Type": "application/json", "authorization": authorizationHeader}

# Create or get Tenant id
tenantResult = session.post(f"{ehrUrl}/tenants", json=tenant, headers=headers)
result = session.get(f"{ehrUrl}/tenants", headers=headers, params={'name': tenant['nameDisplay']})
print(result.content)
tenant = json_result_dict(result)

tenantBaseUrl = f"{ehrUrl}/tenants/{tenant.get('id')}"

# Create or get Registry id
session.post(f"{ehrUrl}/registry", json=registry, headers=headers)
registryResult = session.get(f"{ehrUrl}/registry", headers=headers, params={'name': registry['name']})
registry = json_result_dict(registryResult)
registryId = registry['id']
print(registryId, registry)


b'{"id":53,"nameDisplay":"test","alias":null}'
102 {'id': 102, 'name': 'test script', 'iisHl7Url': 'http://localhost:8080/iis/soap', 'iisFhirUrl': 'http://localhost:8080/iis/fhir', 'iisFhirMessagingUrl': 'http://localhost:8080/iis/fhirMessaging/soap', 'iisUsername': 'test', 'iisFacilityId': 'test', 'iisPassword': 'test', 'description': '', 'receivingFacility': '', 'headers': '', 'default': False}


In [128]:
#  VXU send
vxu = """MSH|^~\&|EHR Sandbox v1.2.3-SNAPSHOT-7|^PH50002^|||20250627023101-0400||VXU^V04^VXU_V04|17510490613994|P|2.5.1|||ER|AL|||||Z22^CDCPHINVS|Volkman-Terry
PID|1||c7350749-3896-c455-8efc-4d4bf96025b5^^^http://hospital.smarthealthit.org^MR||TillmanNIST^DarenNIST^^^Mr.^^|SheronNIST WelchNIST^^^^^^M|19700331|M||2106-3^White^CDCREC|115 Lubowitz Heights Unit 61^^Aguilar^CO^81020^US^P||^NET^X.400^~^PRN^^^^555^7841677|||||||||2135-2^Hispanic or Latino^CDCREC||N|
PD1|||||||||||||||||||||
ORC|RE|104^IIS|1^Volkman-Terry|||||||||
RXA|0|1|19700707||10^IPV^CVX||||01^Historical information - source unspecified^NIP001||^^^|||||||||CP^Complete^HL70322|A
"""

resultVXU = session.post(f"{tenantBaseUrl}/vxu", headers=headers, data=vxu, params={'registryId': registryId})
print(resultVXU.content)

b'{"id":-1,"rawResult":"MSH|^~\\\\&|IIS Sandbox vnull|test|EHR Sandbox v1.2.3-SNAPSHOT-7||20250627145942-0400||ACK^V04^ACK|20250627145942-0400.17|T|2.5.1|||NE|NE|||||Z23^CDCPHINVS|\\nMSA|AE|17510490613994|\\nERR||RXA^1^3^1^1^1|0^Message accepted^HL70357|E|3^Illogical Value error^HL70533^MQE0257^Vaccination admin date is before or after licensed vaccine range^L|||Vaccination admin date is before or after licensed vaccine range |\\nERR||RXA^1^20^1|0^Message accepted^HL70357|W|Code Not Found^The value \'CP^Complete^HL70322\' at location Field RXA-20 (Completion Status) is not member of the value set [HL70322_IZ]^L|||The value \'CP\\\\S\\\\Complete\\\\S\\\\HL70322\' at location Field RXA-20 (Completion Status) is not member of the value set [HL70322_IZ] (reported by NIST)|\\nERR||RXA^1^20^1|0^Message accepted^HL70357|W|Unescaped Separator^The primitive Field RXA-20 (Completion Status) contains at least one unescaped delimiter^L|||The primitive Field RXA-20 (Completion Status) contains at l

In [129]:
#  Qpd/QBP send
vxu = """MSH|^~\&|EHR Sandbox v1.2.3-SNAPSHOT-7|^PH50002^|||20250627024822-0400||QBP^Q11^QBP_Q11|1751050102472.5|P|2.5.1|||ER|AL|||||Z34^CDCPHINVS|
QPD|Z34^Request Immunization History^CDCPHINVS|1751050102472.5|dd0db515-a55d-f88e-9491-5c9ce2dfca71^^^http://hospital.smarthealthit.org^MR|GulgowskiNIST^KristieNIST^DeeNIST^^Mrs.^^|ShemikaNIST HandNIST^^^^^^M|19930713|F|125 Pacocha Stravenue Unit 44^^Colorado Springs^CO^80904^US^P|^NET^X.400^|N||||
RCP|I|20^RD&Records&HL70126|
"""

resultQBP = session.post(f"{tenantBaseUrl}/qbp", headers=headers, data=vxu, params={'registryId': registryId})
print(resultQBP.content)

b'{"id":-1,"rawResult":"MSH|^~\\\\&||IIS Sandbox vnull|EHR Sandbox v1.2.3-SNAPSHOT-7||20250627025943-0400||ACK^V04^ACK|175105078383317|P|2.5.1|||NE|NE|||||Z23^CDCPHINVS\\nMSA|AE|1751050102472.5\\nERR|||101^Required field missing^HL70357|E||||Internal error prevented processing: Cannot invoke \\"org.immregistries.iis.kernal.model.ModelAddress.getAddressLine1()\\" because the return value of \\"org.immregistries.iis.kernal.model.PatientMaster.getFirstAddress()\\" is null (reported by IIS)|\\n","rawSource":"MSH|^~\\\\&|EHR Sandbox v1.2.3-SNAPSHOT-7|^PH50002^|||20250627024822-0400||QBP^Q11^QBP_Q11|1751050102472.5|P|2.5.1|||ER|AL|||||Z34^CDCPHINVS|\\nQPD|Z34^Request Immunization History^CDCPHINVS|1751050102472.5|dd0db515-a55d-f88e-9491-5c9ce2dfca71^^^http://hospital.smarthealthit.org^MR|GulgowskiNIST^KristieNIST^DeeNIST^^Mrs.^^|ShemikaNIST HandNIST^^^^^^M|19930713|F|125 Pacocha Stravenue Unit 44^^Colorado Springs^CO^80904^US^P|^NET^X.400^|N||||\\nRCP|I|20^RD&Records&HL70126|\\n","messageId"

In [133]:
# FHIR Update or Create
fhirResource = """{
  "resourceType": "Patient",
  "identifier": [{
    "type": {
      "coding": [ {
        "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
        "code": "MR"
      } ]
    },
    "system": "http://hospital.smarthealthit.org",
    "value": "dd0db515-a55d-f88e-9491-5c9ce2dfca71"
  }],
  "name": [ {
    "family": "GulgowskiNIST",
    "given": [ "KristieNIST", "DeeNIST" ],
    "prefix": [ "Mrs." ]
  }],
  "gender": "female",
  "birthDate": "1993-07-13",
  "managingOrganization": {
    "identifier": {
      "system": "ehr-sandbox/facility",
      "value": "1"
    }
  }
}"""

fhirResult = session.put(f"{tenantBaseUrl}/fhir-client", headers=headers, data=fhirResource, params={'registryId': registryId, 'type': "Patient"})
print(fhirResult)
print(fhirResult.content)

<Response [200]>
b'{"created":true,"id":"http://localhost:8080/iis/fhir/test/Patient/656/_history/1","operationOutcome":"","errorMessage":""}'


In [132]:
# FHIR Create
fhirResource = """{
  "resourceType": "Immunization",
  "status": "completed",
  "vaccineCode": {
    "coding": [ {
      "system": "http://hl7.org/fhir/sid/cvx",
      "code": "08",
      "display": "Hep B, adolescent or pediatric"
    } ]
  },
  "patient": {
    "identifier": {
      "type": {
        "coding": [ {
          "system": "http://terminology.hl7.org/CodeSystem/v2-0203",
          "code": "MR"
        } ]
      },
      "system": "http://hospital.smarthealthit.org",
      "value": "dd0db515-a55d-f88e-9491-5c9ce2dfca71"
    }
  },
  "occurrenceDateTime": "1993-07-13T02:17:47-04:00",
  "primarySource": false
}"""

# This time no need for type specification
fhirResult = session.post(f"{tenantBaseUrl}/fhir-client", headers=headers, data=fhirResource, params={'registryId': registryId, 'type': "Immunization"})
print(fhirResult)
print(fhirResult.content)

<Response [200]>
b'{"created":true,"id":"http://localhost:8080/iis/fhir/test/Immunization/655/_history/1","operationOutcome":"","errorMessage":""}'
